3. Segment Comparison & Diagnosis
Membandingkan performa antar-shipping mode, region, market. Periksa volume, late rate, dan rata-rata hari keterlambatan secara bersamaan.


In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
data_path = Path("../../Data/Processed/dataco_cleaned.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Processed dataset not found{data_path}\n"
    )

df = pd.read_csv(
    data_path,
    low_memory=False,
    parse_dates= [
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ],
)

print(f"loaded file: {data_path}")
print(f"rows:{len(df):,}")
print(f"columns: {df.shape[1]}")


loaded file: ../../Data/Processed/dataco_cleaned.csv
rows:180,519
columns: 64


In [ ]:
#delivery erformance by shipping mode

shipping_analysis = df[df["delivery_timing"].isin(["Delayed", "Not delayed", "Canceled"])].copy()

shipping_summary = (
    shipping_analysis.groupby("Shipping Mode")
    .agg(total_items=("Order Item Id", "count"),
        delayed_items=("delivery_timing",lambda s: (s == "Delayed").sum()),
        on_time_or_early_items=("delivery_timing",lambda s: (s== "Not delayed").sum()),
        canceled_items=("delivery_timing",lambda s: (s == "Canceled").sum())
    ).reset_index()
)
eligible_items = (shipping_summary["delayed_items"]+shipping_summary["on_time_or_early_items"])
shipping_summary["delay_rate_pct"] = (shipping_summary["delayed_items"] / eligible_items * 100)
shipping_summary["cancellation_rate_pct"] = (shipping_summary["canceled_items"]/shipping_summary["total_items"]*100)
shipping_summary = shipping_summary.sort_values("delay_rate_pct",ascending=False)

display(shipping_summary.round(2))

,Shipping Mode,total_items,delayed_items,on_time_or_early_items,canceled_items,delay_rate_pct,cancellation_rate_pct
0,First Class,27814,26513,0,1301,100.00,4.68
2,Second Class,35216,26987,6819,1410,79.83,4.00
1,Same Day,9737,4454,4839,444,47.93,4.56
3,Standard Class,107752,41023,62130,4599,39.77,4.27


Temuan:
1, First class : 26.513 item terlambat dan tidak ada item tepat waktu(not delayed)
2, Second class : tertinggi ke dua delay rate di 79.83%.
3, Standart class : Jumlah item terbesar (107,752), tapi delay rate lebih rendah (39.77%)
4, Cancelation rate : sekitar 4.00% - 4.68%, relatif berdekatan (perbedaan delay rate lebih besar daripada cancelation rate)

In [ ]:
# checking variance per shipping mode

shipping_diagnostic = (
    shipping_analysis.groupby("Shipping Mode").agg(
        avg_actual_days = ("Days for shipping (real)", "mean"),
        avg_scheduled_days = ("Days for shipment (scheduled)", "mean"),
        min_variance = ("schedule_variance_days","min"),
        max_variance = ("schedule_variance_days", "max"),
        variance_zero_or_negative = ("schedule_variance_days",lambda s: (s <= 0).sum())
    ).reset_index()
)

display(shipping_diagnostic.round(2))

,Shipping Mode,avg_actual_days,avg_scheduled_days,min_variance,max_variance,variance_zero_or_negative
0,First Class,2.00,1.0,1,1,0
1,Same Day,0.48,0.0,0,1,5080
2,Second Class,3.99,2.0,0,4,7138
3,Standard Class,4.00,4.0,-2,2,64901


First class : varians konstan di 1 hari tidak ada pengiriman yang tepat waktu.
Same day : waktu pengiriman 0.48 hari (kurang dari 2 jam)
Second class : rata rata aktual pengiriman 3.99 hari ini menunjukkan delay 2x lipat dari jadwal yaitu 2 hari. 
Standart class : jadwal yang ditetapkan 4 hari dan rata rata sampai dalam 4 hari


In [ ]:
#delivery performance by market 

market_summary = (
    shipping_analysis.groupby("Market").agg(
        total_items=("Order Item Id", "count"),
        delayed_items = ("delivery_timing", lambda s: (s == "Delayed").sum()),
        on_time_or_early_items=("delivery_timing", lambda s: (s == "Not delayed").sum()),
        canceled_items = ("delivery_timing", lambda s : (s =="Canceled").sum())
    ).reset_index()
)

market_summary["eligible_items"] = (
    market_summary["delayed_items"]+market_summary["on_time_or_early_items"]
)

market_summary["delay_rate_pct"]=(
    market_summary["delayed_items"] /
    market_summary["eligible_items"]*100)

market_summary["cancellation_rate_pct"] = (
    market_summary["canceled_items"]/market_summary["total_items"] * 100
)

display(
    market_summary.sort_values("delay_rate_pct", ascending=False).round(2)
)

,Market,total_items,delayed_items,on_time_or_early_items,canceled_items,eligible_items,delay_rate_pct,cancellation_rate_pct
1,Europe,50252,27743,20347,2162,48090,57.69,4.30
4,USCA,25799,14138,10489,1172,24627,57.41,4.54
3,Pacific Asia,41260,22712,16873,1675,39585,57.38,4.06
2,LATAM,51594,28044,21265,2285,49309,56.87,4.43
0,Africa,11614,6340,4814,460,11154,56.84,3.96


Kelima nya memiliki delay rate yang relatif dekat 56,84% - 57.69% :
1, Eropa : delay rate tertinggi 57,69% dengan item terlambat 27,743.
2, USCA : delay rate 57,41 dan item terlambat 14,138.
3, Asia Pasifik : 57,38% dan 22,712.
4, LATAM : 56,87% tetapi jumlah item terlambat terbesar 28,044.
5, Afrika : delay rate terendah 56,84% dan 63,40 item terlambat. 

In [ ]:
# delivery performance by order region 

region_summary = (
    shipping_analysis.groupby("Order Region")
    .agg(
        total_items = ("Order Item Id", "count"),
        delayed_items = ("delivery_timing",lambda s: (s == "Delayed").sum()),
        on_time_or_early_items=("delivery_timing",lambda s: (s == "Not delayed").sum()),
        canceled_items=("delivery_timing",lambda s: (s == "Canceled").sum())
    ).reset_index()
)

region_summary["eligible_items"] = (
    region_summary["delayed_items"]+ region_summary["on_time_or_early_items"]
)
region_summary["delay_rate_pct"] = (
    region_summary["delayed_items"] / region_summary["eligible_items"]* 100
)
region_summary["cancellation_rate_pct"] = (region_summary["canceled_items"]/ region_summary["total_items"]* 100)
display(
    region_summary.sort_values("delay_rate_pct", ascending=False).round(2)
)


,Order Region,total_items,delayed_items,on_time_or_early_items,canceled_items,eligible_items,delay_rate_pct,cancellation_rate_pct
2,Central Africa,1677,972,644,61,1616,60.15,3.64
22,Western Europe,27109,15140,10727,1242,25867,58.53,4.58
14,South of USA,4045,2256,1599,190,3855,58.52,4.70
13,South Asia,7731,4350,3105,276,7455,58.35,3.57
5,East Africa,1852,1036,745,71,1781,58.17,3.83
6,East of USA,6915,3849,2768,298,6617,58.17,4.31
15,Southeast Asia,9539,5297,3839,403,9136,57.98,4.22
20,West Asia,6009,3322,2424,263,5746,57.81,4.38
8,Eastern Europe,3920,2182,1603,135,3785,57.65,3.44
18,US Center,5887,3252,2401,234,5653,57.53,3.97


Central afrika : delay rate tertinggi 60,15% tetapi jumlah item hanya 1,677
western europe : rate 58,53% dan volume 27.109 item
South of usa : rate 58,52% dan volume 4,045 item 
south asia : rate 58,53% dan volume 7.731 item.
West of usa : delay rate paling rendah 51.60% dan tetapi jumlahnya hanya 959 item